In [13]:
import argparse
from transformers import AutoTokenizer
from adapters import AutoAdapterModel
from corpus_io import specter2_paper_text
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import torch
import os
import itertools
from tqdm import tqdm
import datasets
import json
import pickle
import glob
import zipfile
from sklearn.metrics.pairwise import cosine_similarity as cos
from collections import defaultdict


def _existing_path(candidates):
    for p in candidates:
        if p and os.path.isfile(p):
            return p
    return None


def _load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def _load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def _load_from_official_zip(member_name, zip_path='./.cache/datasets/emnlp407.data.zip', is_pickle=False):
    if not os.path.isfile(zip_path):
        return None
    with zipfile.ZipFile(zip_path) as z:
        if member_name not in z.namelist():
            return None
        raw = z.read(member_name)
        if is_pickle:
            return pickle.loads(raw)
        return json.loads(raw.decode('utf-8'))


def load_eval_queries(dataset, data_dir, corpus_with_labels):
    ds = dataset.lower().strip()

    if ds == 'litsearch':
        raw = datasets.load_dataset('princeton-nlp/LitSearch', 'query', split='full')
        return [{'query': q['query'], 'corpusids': [str(i) for i in q['corpusids']]} for q in raw]

    if ds == 'csfcube':
        ann_files = [
            _existing_path([
                os.path.join(data_dir, f'test-pid2anns-csfcube-{aspect}.json'),
                os.path.join(data_dir, 'Dataset', 'CSFCube', f'test-pid2anns-csfcube-{aspect}.json'),
            ])
            for aspect in ('background', 'method', 'result')
        ]

        ann_data = []
        for f in ann_files:
            if f:
                ann_data.append(_load_json(f))
        if not ann_data:
            # fallback to official zip in repo cache
            for aspect in ('background', 'method', 'result'):
                d = _load_from_official_zip(f'Dataset/CSFCube/test-pid2anns-csfcube-{aspect}.json', is_pickle=False)
                if d is not None:
                    ann_data.append(d)

        if not ann_data:
            raise FileNotFoundError(
                'Cannot find CSFCube test annotation files. Expected test-pid2anns-csfcube-*.json '
                'under data_dir (or Dataset/CSFCube), or ./.cache/datasets/emnlp407.data.zip.'
            )

        qid2rels = defaultdict(set)
        for anns in ann_data:
            for qid, rec in anns.items():
                cands = rec.get('cands', [])
                rel = rec.get('relevance_adju') or rec.get('relevance_max') or []
                for cid, score in zip(cands, rel):
                    # Graded 0–3; Kang et al. / paper Table 1 (doc/query≈13.32) use relevance ≥ 2.
                    if int(score) >= 2:
                        qid2rels[str(qid)].add(str(cid))

        topic_meta = {}
        _tp = _existing_path([os.path.join(data_dir, 'specter2_topics.json')])
        if _tp:
            topic_meta = _load_json(_tp)

        queries = []
        for qid in sorted(qid2rels.keys()):
            doc = corpus_with_labels.get(str(qid))
            if not doc:
                continue
            meta = topic_meta.get(str(qid))
            if meta and (meta.get('title') or meta.get('abstract')):
                qtxt = specter2_paper_text(meta.get('title'), meta.get('abstract')).strip()
            elif doc.get('title') or doc.get('abstract'):
                qtxt = specter2_paper_text(doc.get('title'), doc.get('abstract')).strip()
            else:
                qtxt = (doc.get('text') or '').strip()
            if not qtxt:
                continue
            queries.append({'query': qtxt, 'corpusids': sorted(qid2rels[qid])})
        return queries

    if ds == 'dorismae':
        q_path = _existing_path([
            os.path.join(data_dir, 'queries'),
            os.path.join(data_dir, 'Dataset', 'DORISMAE', 'queries'),
        ])
        r_path = _existing_path([
            os.path.join(data_dir, 'qrels'),
            os.path.join(data_dir, 'Dataset', 'DORISMAE', 'qrels'),
        ])

        if q_path and r_path:
            q_dict = _load_pickle(q_path)
            r_dict = _load_pickle(r_path)
        else:
            q_dict = _load_from_official_zip('Dataset/DORISMAE/queries', is_pickle=True)
            r_dict = _load_from_official_zip('Dataset/DORISMAE/qrels', is_pickle=True)
            if q_dict is None or r_dict is None:
                raise FileNotFoundError(
                    'Cannot find DORISMAE queries/qrels. Expected files under data_dir '
                    '(or Dataset/DORISMAE), or ./.cache/datasets/emnlp407.data.zip.'
                )

        queries = []
        for qid, qtxt in q_dict.items():
            rels = r_dict.get(qid, {})
            rel_ids = [str(cid) for cid, score in rels.items() if int(score) > 0]
            queries.append({'query': str(qtxt), 'corpusids': rel_ids})
        return queries

    raise ValueError(f'Unknown dataset: {dataset}')

In [14]:
# options: 'litsearch', 'csfcube', 'dorismae'
dataset = 'csfcube'

data_dir_by_dataset = {
    'litsearch': './LitSearch',
    'csfcube': './CSFCube',
    'dorismae': './DORISMAE',
}
data_dir = data_dir_by_dataset[dataset]
print('dataset:', dataset, 'data_dir:', data_dir)

dataset: csfcube data_dir: ./CSFCube


In [17]:
import os
import requests
import json

base = os.environ.get("TAMUS_AI_CHAT_API_ENDPOINT", "https://chat-api.tamu.ai").rstrip("/")
key = os.environ.get("TAMUS_AI_CHAT_API_KEY", "").strip()

url = f"{base}/api/chat/completions"
headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json",
}

payload = {
    "model": "protected.gemini-2.0-flash-lite",
    "stream": 0,
    "messages": [
        {"role": "user", "content": "Why is the sky blue?"}
    ]
}

print("POST", url)
print("payload =", json.dumps(payload, indent=2))

try:
    resp = requests.post(url, headers=headers, json=payload, timeout=(10, 180))
    print("status =", resp.status_code)
    print("content-type =", resp.headers.get("content-type"))
    print("raw text preview =")
    print(resp.text[:2000])
except Exception as e:
    print("request failed:", repr(e))

POST https://chat-api.tamu.ai/api/chat/completions
payload = {
  "model": "protected.gemini-2.0-flash-lite",
  "stream": 0,
  "messages": [
    {
      "role": "user",
      "content": "Why is the sky blue?"
    }
  ]
}
status = 500
content-type = text/plain; charset=utf-8
raw text preview =
Internal Server Error


In [3]:
import os
import requests
import json

base = os.environ.get("TAMUS_AI_CHAT_API_ENDPOINT", "https://chat-api.tamu.ai").rstrip("/")
key = os.environ.get("TAMUS_AI_CHAT_API_KEY", "").strip()

url = f"{base}/api/chat/completions"
headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

payload = {
    "model": "protected.gpt-4.1",
    "stream": False,
    "messages": [
        {"role": "user", "content": "Reply with exactly ALIVE."}
    ]
}

print("POST", url)
print("payload =", json.dumps(payload, indent=2))

def extract_sse_text(raw_text: str) -> str:
    chunks = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line.startswith("data:"):
            continue
        data_str = line[len("data:"):].strip()
        if not data_str or data_str == "[DONE]":
            continue
        try:
            obj = json.loads(data_str)
        except Exception:
            continue

        if "choices" in obj:
            for choice in obj["choices"]:
                delta = choice.get("delta", {})
                if "content" in delta:
                    chunks.append(delta["content"])
                message = choice.get("message", {})
                if "content" in message:
                    chunks.append(message["content"])
    return "".join(chunks).strip()

try:
    resp = requests.post(url, headers=headers, json=payload, timeout=(10, 180))
    print("status =", resp.status_code)
    ctype = resp.headers.get("content-type", "")
    print("content-type =", ctype)
    print("raw text preview =")
    print(resp.text[:2000])

    if "text/event-stream" in ctype:
        extracted = extract_sse_text(resp.text)
        print("\nextracted text =")
        print(repr(extracted))
    else:
        try:
            data = resp.json()
            print("\nparsed json =")
            print(json.dumps(data, indent=2)[:4000])
        except Exception as e:
            print("\njson parse failed:", repr(e))

except Exception as e:
    print("request failed:", repr(e))

POST https://chat-api.tamu.ai/api/chat/completions
payload = {
  "model": "protected.gpt-4.1",
  "stream": false,
  "messages": [
    {
      "role": "user",
      "content": "Reply with exactly ALIVE."
    }
  ]
}
status = 500
content-type = text/plain; charset=utf-8
raw text preview =
Internal Server Error

json parse failed: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')


In [4]:
id2label, label2id, id2name, name2id = {}, {}, [], {}
with open('classifier/labels.txt') as f:
    for i, line in enumerate(f):
        label, name, _ = line.strip().split('\t')
        id2label[i] = label
        label2id[label] = i
        id2name.append(name)
        name2id[name] = i

corpus_with_labels = json.load(open(f'{data_dir}/specter2_corpus_with-topic-terms.json'))
corpus_embeddings = torch.load(f'{data_dir}/corpus-enc-specter.pt')
id2corpus_id = pickle.load(open(f'{data_dir}/corpus-enc-index.pkl', 'rb'))

# Normalize corpus embeddings shape to [num_docs, hidden_dim]
if isinstance(corpus_embeddings, list):
    corpus_embeddings = torch.cat(corpus_embeddings, dim=0)
if isinstance(corpus_embeddings, torch.Tensor):
    corpus_embeddings = corpus_embeddings.numpy()

raw_queries = load_eval_queries(dataset, data_dir, corpus_with_labels)
print('num_eval_queries:', len(raw_queries))

num_eval_queries: 34


In [5]:
(id2phrase, phrase2id) = pickle.load(open(f'{data_dir}/specter2_corpus_with-topic-terms.json.phrase_idx.pkl', 'rb'))
phrase_embeddings = torch.load(f'{data_dir}/specter2_corpus_with-topic-terms.json.phrase-enc-specter.pt', weights_only=False)
topic_embeddings = torch.load(f'{data_dir}/topic-enc-specter.pt', weights_only=False)

In [6]:
gpu_num = 0
if torch.cuda.is_available():
    device = torch.device(f'cuda:{gpu_num}')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

backbone = 'allenai/specter2_base'
tokenizer = AutoTokenizer.from_pretrained(backbone)
encoder = AutoAdapterModel.from_pretrained(backbone)
encoder.load_adapter('allenai/specter2', source='hf', load_as='proximity')
encoder.set_active_adapters('proximity')
encoder.to(device)

Using device: mps


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 64527.75it/s]
There are adapters available but none are activated for the forward pass.


BertAdapterModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttentionWithAdapters(
              (query): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (key): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (value): LoRALinearTorch(
             

In [7]:
all_embeddings = torch.cat((topic_embeddings, phrase_embeddings), dim=0).numpy()
all_embeddings = all_embeddings / np.linalg.norm(all_embeddings, axis=1, keepdims=True)
all_terms = id2name + id2phrase
all_term2id = {t:i for i,t in enumerate(all_terms)}

doc2term_id = {}
for corpusid in corpus_with_labels:
    doc_terms = [t[1] for t in corpus_with_labels[corpusid]['topics']] + \
        [t.lower() for t in corpus_with_labels[corpusid]['terms']]
    doc2term_id[corpusid] = set([all_term2id[t] for t in doc_terms if t in all_term2id and t != ''])

# Global Aspect Weighting (GAW): idf-like rarity prior over aspects.
num_docs = len(doc2term_id)
term_doc_freq = np.zeros(len(all_terms), dtype=np.float32)
for term_ids in doc2term_id.values():
    if len(term_ids) == 0:
        continue
    term_doc_freq[list(term_ids)] += 1

# Higher weight for rarer terms, lower for frequent terms.
gaw_weights = np.log((num_docs + 1.0) / (term_doc_freq + 1.0)).astype(np.float32)
gaw_weights = np.maximum(gaw_weights, 1e-6)

In [8]:
import os
import time
import requests

_PLACEHOLDERS = {'paste-your-real-openai-key-here', 'paste-your-tamu-key-here', '', 'None', 'none'}

# Always re-read .env on every cell run so stale kernel values don't win
def _load_dotenv(path='.env'):
    if not os.path.isfile(path):
        return
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            k, v = line.split('=', 1)
            k, v = k.strip(), v.strip()
            if k and v and v not in _PLACEHOLDERS:
                os.environ[k] = v
_load_dotenv('.env')

api_provider = os.environ.get('CHAT_API_PROVIDER', 'tamu').strip().lower()
SKIP_LLM = os.environ.get('SKIP_LLM', '').strip().lower() in ('1', 'true', 'yes')

# Inline fallback - paste key here if .env is not working
tamu_api_key = ''
if tamu_api_key:
    os.environ['TAMUS_AI_CHAT_API_KEY'] = tamu_api_key


def _tamu_base() -> str:
    return (os.environ.get('TAMUS_AI_CHAT_API_ENDPOINT') or 'https://chat-api.tamu.ai').rstrip('/')


def _tamu_completions_url() -> str:
    return f'{_tamu_base()}/api/chat/completions'


def _resolve_tamu_model(model_name: str) -> str:
    # If caller already passed a TAMU-native model name, use it directly
    if model_name.startswith('protected.') or model_name.startswith('tamu.'):
        return model_name
    return os.environ.get('TAMU_DEFAULT_MODEL', 'protected.gemini-2.0-flash-lite')


def _parse_sse(raw_text: str) -> str:
    # Parse SSE lines and concatenate delta content chunks
    parts = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line.startswith('data:'):
            continue
        payload = line[len('data:'):].strip()
        if payload == '[DONE]':
            break
        try:
            chunk = json.loads(payload)
            choices = chunk.get('choices') or []
            if choices:
                delta = (choices[0].get('delta') or {})
                content = delta.get('content')
                if content:
                    parts.append(content)
        except Exception:
            continue
    return ''.join(parts)


if not SKIP_LLM:
    if api_provider == 'tamu':
        tamu_key = (os.environ.get('TAMUS_AI_CHAT_API_KEY') or '').strip()
        if not tamu_key or tamu_key in _PLACEHOLDERS:
            raise ValueError(
                '\n\n*** TAMUS_AI_CHAT_API_KEY is missing or still a placeholder. ***\n'
                'Open .env in the project root and paste your real TAMU portal token, '
                'then re-run THIS cell. Or export the variable before launching Jupyter. '
                'Set SKIP_LLM=1 to run dense-only with no LLM.\n'
            )
    else:
        oai_key = (os.environ.get('OPENAI_API_KEY') or '').strip()
        if not oai_key or oai_key in _PLACEHOLDERS:
            raise ValueError(
                '\n\n*** OPENAI_API_KEY is missing or still a placeholder. ***\n'
                'Open .env and paste your real OpenAI key (starts with sk-), then re-run THIS cell.\n'
            )


_api_call_warned = set()
_tamu_payload_printed = False

def api_call(doc, instruction=None, temperature=0.2, model='gpt-4.1-mini'):
    global _tamu_payload_printed
    if SKIP_LLM:
        return ''
    if instruction:
        messages = [{'role': 'system', 'content': instruction}, {'role': 'user', 'content': doc}]
    else:
        messages = [{'role': 'user', 'content': doc}]

    if api_provider == 'tamu':
        url = _tamu_completions_url()
        key = (os.environ.get('TAMUS_AI_CHAT_API_KEY') or '').strip()
        headers = {'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'}
        req_model = _resolve_tamu_model(model)
    else:
        url = 'https://api.openai.com/v1/chat/completions'
        key = (os.environ.get('OPENAI_API_KEY') or '').strip()
        headers = {'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'}
        req_model = model

    payload = {
        'model': req_model,
        'messages': messages,
        'temperature': temperature,
        'stream': False,
    }

    if api_provider == 'tamu' and not _tamu_payload_printed:
        print(f'[api_call DEBUG] TAMU payload (first call only): {json.dumps(payload, indent=2)[:400]}')
        _tamu_payload_printed = True

    last_status, last_body = None, None
    for attempt in range(5):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
            last_status = resp.status_code
            raw_text = resp.text or ''
            last_body = raw_text[:300]

            ct = resp.headers.get('content-type', '').lower()

            if resp.status_code in (429, 500, 502, 503, 504):
                time.sleep(3 + attempt)
                continue

            if resp.status_code != 200:
                warn_key = f'{resp.status_code}'
                if warn_key not in _api_call_warned:
                    print(f'\n[api_call WARNING] HTTP {resp.status_code} from {url}')
                    print(f'  response: {last_body}')
                    print(f'  provider={api_provider}  model={req_model}')
                    _api_call_warned.add(warn_key)
                return ''

            # TAMUS returns text/event-stream even when stream=false is requested
            if 'text/event-stream' in ct:
                extracted = _parse_sse(raw_text)
                if extracted:
                    return extracted
                time.sleep(1 + attempt)
                continue

            # Normal JSON path (OpenAI or future TAMUS non-streaming)
            if not raw_text.strip():
                time.sleep(2 + attempt)
                continue
            data = resp.json()
            if isinstance(data, dict) and data.get('choices'):
                c0 = data['choices'][0]
                return str((c0.get('message') or {}).get('content') or '')
            err = (data or {}).get('error') if isinstance(data, dict) else None
            msg = str((err or {}).get('message') or data)
            if 'rate' in msg.lower() or 'quota' in msg.lower():
                time.sleep(3 + attempt)
                continue
            warn_key = 'bad_json'
            if warn_key not in _api_call_warned:
                print(f'\n[api_call WARNING] unexpected response body: {last_body}')
                _api_call_warned.add(warn_key)
            return ''

        except Exception as exc:
            last_body = str(exc)
            time.sleep(2 + attempt)
            continue

    warn_key = 'timeout'
    if warn_key not in _api_call_warned:
        print(f'\n[api_call WARNING] all 5 attempts failed. last_status={last_status} last_body={last_body}')
        _api_call_warned.add(warn_key)
    return ''

In [9]:
print(f'SKIP_LLM    : {SKIP_LLM}')
print(f'api_provider: {api_provider}')

if api_provider == 'tamu':
    _k = (os.environ.get('TAMUS_AI_CHAT_API_KEY') or '').strip()
    print(f'TAMUS key   : {"SET (len=" + str(len(_k)) + ", starts=" + repr(_k[:6]) + ")" if _k else "MISSING"}')
    print(f'TAMUS base  : {_tamu_base()}')
    print(f'default model: {os.environ.get("TAMU_DEFAULT_MODEL", "protected.gemini-2.0-flash-lite")}')
else:
    _k = (os.environ.get('OPENAI_API_KEY') or '').strip()
    print(f'OPENAI key  : {"SET (len=" + str(len(_k)) + ")" if _k else "MISSING"}')

print()
if not SKIP_LLM:
    _api_call_warned.clear()
    global _tamu_payload_printed
    _tamu_payload_printed = False
    _tout = api_call('Reply with exactly the single word: ALIVE')
    if _tout:
        print(f'API works -> response={repr(_tout[:200])}')
    else:
        print('API FAILED - returned empty string. Check warnings above.')
else:
    print('SKIP_LLM=True: LLM calls disabled.')

SKIP_LLM    : False
api_provider: tamu
TAMUS key   : SET (len=35, starts='sk-4f9')
TAMUS base  : https://chat-api.tamu.ai
default model: protected.gemini-2.0-flash-lite

[api_call DEBUG] TAMU payload (first call only): {
  "model": "protected.gemini-2.0-flash-lite",
  "messages": [
    {
      "role": "user",
      "content": "Reply with exactly the single word: ALIVE"
    }
  ],
  "temperature": 0.2,
  "stream": false
}


KeyboardInterrupt: 

In [ ]:
def evaluate_metrics(results, ks=(50, 100), verbose=False):
    """
    Evaluate Hit@k and Recall@k for the given ks (default R@50, R@100 for paper-style tables).

    Returns metrics dict with 'hit@k' and 'recall@k' keyed by k.
    """
    ks = list(ks)
    total_hits_at_k = {k: 0.0 for k in ks}
    total_recall_at_k = {k: 0.0 for k in ks}
    num_queries = 0

    for r in results:
        ground_truth = set(r['corpusids'])
        retrieved = [i for i, _ in r['retrieved']]

        num_relevant = len(ground_truth)
        if num_relevant == 0:
            continue

        num_queries += 1

        for k in ks:
            retrieved_at_k = retrieved[:k]
            retrieved_set = set(retrieved_at_k)
            hit = 1.0 if len(ground_truth & retrieved_set) > 0 else 0.0
            total_hits_at_k[k] += hit
            num_relevant_at_k = len(ground_truth & retrieved_set)
            recall_at_k = num_relevant_at_k / num_relevant
            total_recall_at_k[k] += recall_at_k

    average_hits_at_k = {k: total_hits_at_k[k] / num_queries if num_queries > 0 else 0.0 for k in ks}
    average_recall_at_k = {k: total_recall_at_k[k] / num_queries if num_queries > 0 else 0.0 for k in ks}

    if verbose:
        print(f'Total Queries Evaluated: {num_queries}')
        for k in ks:
            print(f'\nMetrics at {k}:')
            print(f'Hit@{k}: {average_hits_at_k[k]:.4f}')
            print(f'Recall@{k}: {average_recall_at_k[k]:.4f}')

    return {
        'hit@k': average_hits_at_k,
        'recall@k': average_recall_at_k,
        'num_queries': num_queries,
    }


# Paper-style cutoffs (CSFCube / DORISMAE / LitSearch): R@50, R@100
EVAL_KS = (50, 100)


def print_recall_table(dataset_name, e_uniform, e_gaw):
    """One-line summary: uniform vs GAW, R@50 and R@100."""
    ru = e_uniform['recall@k']
    rg = e_gaw['recall@k']
    print(f'\n=== {dataset_name} (queries={e_uniform["num_queries"]}) ===')
    print(f'{"":12}  {"R@50":>10}  {"R@100":>10}')
    print(f'{"uniform":12}  {ru[50]:>10.4f}  {ru[100]:>10.4f}')
    print(f'{"gaw":12}  {rg[50]:>10.4f}  {rg[100]:>10.4f}')


def print_recall_semrank_table(dataset_name, e_uniform, e_gaw):
    """SemRank uniform vs GAW on the same qrels."""
    ru, rg = e_uniform['recall@k'], e_gaw['recall@k']
    nq = e_uniform['num_queries']
    print(f'\n=== {dataset_name} (queries={nq}) — SemRank ===')
    print(f'{"":14}  {"R@50":>10}  {"R@100":>10}')
    print(f'{"semrank+u":14}  {ru[50]:>10.4f}  {ru[100]:>10.4f}')
    print(f'{"semrank_gaw":14}  {rg[50]:>10.4f}  {rg[100]:>10.4f}')
    ds = dataset_name.lower().strip()
    if ds == 'csfcube':
        print(
            '\nPaper Table 2 (CSFCube): SPECTER-v2 dense ~0.533 / ~0.686; +SemRank ~0.622 / ~0.760. '
            'Qrels: union of aspect files, relevance score ≥ 2 (matches Table 1 doc/query≈13.32).'
        )

In [ ]:
query_instruction = '''
You will receive a query for computer science research papers and a ranked list of papers returned by a retriever.
You will also be provided a list of research topics and key terms with their frequencies that are frequently mentioned by the top-100 papers returned by the retriever.
Your task is to improve the provided retrieval results by selecting a list of topics and terms that can accurately identify the relevant papers of the query.
Make sure your selection is strictly based on the original query and does not contain repeated concepts.
Output format: '<ans>selection 1, selection 2, ...</ans>'.
Retriever result:
%s
Candidate topics:
%s
Candidate key terms:
%s
Original Query:
%s
'''

In [ ]:
def score_mapping(s, scores):
    std = np.std(scores)
    if std < 1e-12:
        return 0.0
    return (s - np.mean(scores)) / std


def rerank_with_mode(curr_results, selected_concepts, mode='uniform'):
    selected_term_ids = [all_term2id[t] for t in selected_concepts if t in all_term2id]
    if len(selected_term_ids) == 0:
        return curr_results

    all_term_emb = all_embeddings[selected_term_ids]

    if mode == 'gaw':
        term_weights = gaw_weights[selected_term_ids].astype(np.float32)
        term_weights = term_weights / np.sum(term_weights)
    else:
        term_weights = np.ones(len(selected_term_ids), dtype=np.float32) / len(selected_term_ids)

    reranked = []
    for corpusid, base_score in curr_results:
        doc_term_ids = list(doc2term_id[str(corpusid)])
        if len(doc_term_ids) == 0:
            reranked.append((corpusid, 0.0, base_score))
            continue

        doc_embeddings = all_embeddings[doc_term_ids]
        max_sim = np.matmul(all_term_emb, doc_embeddings.T).max(axis=1)
        concept_score = float(np.sum(term_weights * max_sim))
        reranked.append((corpusid, concept_score, base_score))

    all_scores = np.array([s for _, s, _ in reranked])
    reranked = [(i, c + score_mapping(s, all_scores)) for i, s, c in reranked]
    reranked = sorted(reranked, key=lambda x: x[1], reverse=True)
    return reranked


# options: 'uniform', 'gaw', 'both'
scoring_mode = 'both'

# Defaults match upstream github.com/yzhan238/SemRank SemRank.ipynb: top_k=1000, N=100, Nk=50.
RETRIEVAL_TOP_K = 500  # int, or None = entire corpus (slower / more memory per query)
CONCEPT_AGG_TOP_N = 100  # aggregate topic/term counts from top-N dense hits (upstream N=100)
LLM_TOP_PAPERS = 50  # paper snippets in LLM prompt (upstream Nk=50)
LLM_TOP_K_TOPICS = 50  # candidate topic lines (upstream Nk=50)
LLM_TOP_K_TERMS = 50  # candidate key-term lines (upstream Nk=50)

# --- SemRank LLM / rerank debug (set DEBUG_SEMRANK=False to skip) ---
DEBUG_SEMRANK = True
DEBUG_LOG_FIRST_N = 5  # print raw llm + concepts for query_id in [0, N)
DEBUG_LLM_SNIP = 2500  # max chars of llm_output to store per log line
MANUAL_COMPARE_IDX = 0  # print dense vs SemRank top-10 vs qrels for this query index

debug_rows = []
manual_snapshot = None
_n_empty_concepts = 0
_n_top10_diff_u = 0
_n_top10_diff_g = 0

results_uniform = []
results_gaw = []

for query_id, ori_q in enumerate(tqdm(raw_queries)):
    query_text = ori_q['query']

    # base retriever
    inputs = tokenizer(query_text, return_tensors="pt", truncation=True, max_length=512, return_token_type_ids=False).to(device)
    with torch.no_grad():
        output = encoder(**inputs)
    emb = output.last_hidden_state[0, 0, :].cpu().numpy()

    scores = cos(emb[np.newaxis, :], corpus_embeddings)[0]
    ranked_idx = np.argsort(-scores)
    if RETRIEVAL_TOP_K is None:
        top_k = ranked_idx
    else:
        top_k = ranked_idx[: int(RETRIEVAL_TOP_K)]
    curr_results = [(id2corpus_id[i], scores[i]) for i in top_k]
    scores = np.array([s for _, s in curr_results])
    curr_results = [(i, score_mapping(s, scores)) for i, s in curr_results]

    # construct candidate concepts
    topk_terms_dict = defaultdict(int)
    topk_topics_dict = defaultdict(int)
    N = len(curr_results) if CONCEPT_AGG_TOP_N is None else min(int(CONCEPT_AGG_TOP_N), len(curr_results))
    for rank, (corpusid, score) in enumerate(curr_results[:N]):
        for _, topic, _ in corpus_with_labels[str(corpusid)]['topics']:
            topk_topics_dict[topic] += 1
        for term in corpus_with_labels[str(corpusid)]['terms']:
            topk_terms_dict[term.lower()] += 1

    top_terms = [f'{t}, {c}' for t, c in sorted(topk_terms_dict.items(), key=lambda x: x[1], reverse=True)[:LLM_TOP_K_TERMS]]
    top_topics = [f'{t}, {c}' for t, c in sorted(topk_topics_dict.items(), key=lambda x: x[1], reverse=True)[:LLM_TOP_K_TOPICS]]

    top_papers = []
    for rank, (corpusid, _) in enumerate(curr_results[:min(LLM_TOP_PAPERS, len(curr_results))]):
        doc = corpus_with_labels[str(corpusid)]
        title = doc.get('title', '')
        abstract = doc.get('abstract', '')
        if title or abstract:
            paper_txt = f'Title: {title}\nAbstract: {abstract}'
        else:
            paper_txt = doc.get('text', '')
        top_papers.append(f'[{rank + 1}] {paper_txt}')

    llm_input = query_instruction % ('\n'.join(top_papers), '\n'.join(top_topics), '\n'.join(top_terms), query_text)
    llm_output = api_call(llm_input, model='gpt-4.1-mini')

    selected_concepts = []
    if '<ans>' in llm_output and '</ans>' in llm_output:
        selected_concepts = [t.strip().lower() for t in llm_output.split('<ans>')[1].split('</ans>')[0].split(', ')]
        selected_concepts = [t for t in selected_concepts if t in all_term2id]

    if len(selected_concepts) == 0:
        uniform_ranked = curr_results
        gaw_ranked = curr_results
    else:
        uniform_ranked = rerank_with_mode(curr_results, selected_concepts, mode='uniform')
        gaw_ranked = rerank_with_mode(curr_results, selected_concepts, mode='gaw')

    if DEBUG_SEMRANK:
        if len(selected_concepts) == 0:
            _n_empty_concepts += 1
        d10 = [str(x[0]) for x in curr_results[:10]]
        u10 = [str(x[0]) for x in uniform_ranked[:10]]
        g10 = [str(x[0]) for x in gaw_ranked[:10]]
        if d10 != u10:
            _n_top10_diff_u += 1
        if d10 != g10:
            _n_top10_diff_g += 1
        if query_id < DEBUG_LOG_FIRST_N:
            debug_rows.append({
                'query_id': query_id,
                'llm_output': (llm_output or '')[:DEBUG_LLM_SNIP],
                'selected_concepts': list(selected_concepts),
                'n_concepts': len(selected_concepts),
                'dense_top5': d10[:5],
                'uniform_top5': u10[:5],
            })
        if query_id == MANUAL_COMPARE_IDX:
            rel = list(ori_q['corpusids'])
            manual_snapshot = {
                'query_id': query_id,
                'n_relevant': len(rel),
                'qrels_head': rel[:25],
                'dense_top10': d10,
                'uniform_top10': u10,
                'gaw_top10': g10,
                'relevant_in_dense_top10': [i for i in d10 if i in set(rel)],
                'relevant_in_uniform_top10': [i for i in u10 if i in set(rel)],
            }

    if scoring_mode in ['uniform', 'both']:
        results_uniform.append({
            'query': query_text,
            'corpusids': ori_q['corpusids'],
            'retrieved': uniform_ranked,
        })

    if scoring_mode in ['gaw', 'both']:
        results_gaw.append({
            'query': query_text,
            'corpusids': ori_q['corpusids'],
            'retrieved': gaw_ranked,
        })

if scoring_mode == 'uniform':
    results = results_uniform
elif scoring_mode == 'gaw':
    results = results_gaw
else:
    results = {
        'uniform': results_uniform,
        'gaw': results_gaw,
    }

if DEBUG_SEMRANK:
    nq = len(raw_queries)
    print('\n=== SemRank LLM / rerank debug ===')
    print(f'queries={nq} | empty selected_concepts (pure dense fallback): {_n_empty_concepts} ({100*_n_empty_concepts/max(nq,1):.1f}%)')
    print(f'top-10 ranking differs from dense: uniform {_n_top10_diff_u}, gaw {_n_top10_diff_g}')
    for row in debug_rows:
        print(f"\n--- query_id={row['query_id']} | n_concepts={row['n_concepts']} ---")
        print('selected_concepts:', row['selected_concepts'][:40], '...' if len(row['selected_concepts']) > 40 else '')
        print('dense_top5:   ', row['dense_top5'])
        print('uniform_top5: ', row['uniform_top5'])
        print('llm_output (trunc):', row['llm_output'][:1200], '...' if len(row['llm_output']) > 1200 else '')
    if manual_snapshot is not None:
        m = manual_snapshot
        print(f"\n=== MANUAL_COMPARE_IDX={MANUAL_COMPARE_IDX} (query_id={m['query_id']}) ===")
        print(f"|qrels|={m['n_relevant']} (showing first 25 ids): {m['qrels_head']}")
        print('dense_top10:   ', m['dense_top10'])
        print('uniform_top10: ', m['uniform_top10'])
        print('gaw_top10:     ', m['gaw_top10'])
        print('relevant ∩ dense_top10:   ', m['relevant_in_dense_top10'])
        print('relevant ∩ uniform_top10: ', m['relevant_in_uniform_top10'])

  0%|          | 0/34 [00:08<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
if 'EVAL_KS' not in globals():
    EVAL_KS = (50, 100)
if isinstance(results, dict):
    e_uniform = evaluate_metrics(results['uniform'], EVAL_KS, verbose=False)
    e_gaw = evaluate_metrics(results['gaw'], EVAL_KS, verbose=False)
    e = {'uniform': e_uniform, 'gaw': e_gaw}
    print_recall_semrank_table(dataset, e_uniform, e_gaw)
else:
    e_other = evaluate_metrics(results, EVAL_KS, verbose=False)
    e = {'mode': e_other}
    print(f'\n=== {dataset} (queries={e_other["num_queries"]}) — {scoring_mode} ===')
    print(f'{"":14}  {"R@50":>10}  {"R@100":>10}')
    print(f'{scoring_mode:14}  {e_other["recall@k"][50]:>10.4f}  {e_other["recall@k"][100]:>10.4f}')


=== csfcube (queries=34) — SemRank ===
                      R@50       R@100
semrank+u           0.6816      0.7588
semrank_gaw         0.6816      0.7588

Paper Table 2 (CSFCube): SPECTER-v2 dense ~0.533 / ~0.686; +SemRank ~0.622 / ~0.760. Qrels: union of aspect files, relevance score ≥ 2 (matches Table 1 doc/query≈13.32).


In [ ]:
def score_mapping(s, scores):
    std = np.std(scores)
    if std < 1e-12:
        return 0.0
    return (s - np.mean(scores)) / std


def rerank_with_mode(curr_results, selected_concepts, mode='uniform'):
    selected_term_ids = [all_term2id[t] for t in selected_concepts if t in all_term2id]
    if len(selected_term_ids) == 0:
        return curr_results

    all_term_emb = all_embeddings[selected_term_ids]

    if mode == 'gaw':
        term_weights = gaw_weights[selected_term_ids].astype(np.float32)
        term_weights = term_weights / np.sum(term_weights)
    else:
        term_weights = np.ones(len(selected_term_ids), dtype=np.float32) / len(selected_term_ids)

    reranked = []
    for corpusid, base_score in curr_results:
        doc_term_ids = list(doc2term_id[str(corpusid)])
        if len(doc_term_ids) == 0:
            reranked.append((corpusid, 0.0, base_score))
            continue

        doc_embeddings = all_embeddings[doc_term_ids]
        max_sim = np.matmul(all_term_emb, doc_embeddings.T).max(axis=1)
        concept_score = float(np.sum(term_weights * max_sim))
        reranked.append((corpusid, concept_score, base_score))

    all_scores = np.array([s for _, s, _ in reranked])
    reranked = [(i, c + score_mapping(s, all_scores)) for i, s, c in reranked]
    reranked = sorted(reranked, key=lambda x: x[1], reverse=True)
    return reranked


# options: 'uniform', 'gaw', 'both'
scoring_mode = 'both'

# Defaults match upstream github.com/yzhan238/SemRank SemRank.ipynb: top_k=1000, N=100, Nk=50.
RETRIEVAL_TOP_K = 1000  # int, or None = entire corpus (slower / more memory per query)
CONCEPT_AGG_TOP_N = 100  # aggregate topic/term counts from top-N dense hits (upstream N=100)
LLM_TOP_PAPERS = 50  # paper snippets in LLM prompt (upstream Nk=50)
LLM_TOP_K_TOPICS = 50  # candidate topic lines (upstream Nk=50)
LLM_TOP_K_TERMS = 50  # candidate key-term lines (upstream Nk=50)

# --- SemRank LLM / rerank debug (set DEBUG_SEMRANK=False to skip) ---
DEBUG_SEMRANK = True
DEBUG_LOG_FIRST_N = 5  # print raw llm + concepts for query_id in [0, N)
DEBUG_LLM_SNIP = 2500  # max chars of llm_output to store per log line
MANUAL_COMPARE_IDX = 0  # print dense vs SemRank top-10 vs qrels for this query index

debug_rows = []
manual_snapshot = None
_n_empty_concepts = 0
_n_top10_diff_u = 0
_n_top10_diff_g = 0

results_uniform = []
results_gaw = []

for query_id, ori_q in enumerate(tqdm(raw_queries)):
    query_text = ori_q['query']

    # base retriever
    inputs = tokenizer(query_text, return_tensors="pt", truncation=True, max_length=512, return_token_type_ids=False).to(device)
    with torch.no_grad():
        output = encoder(**inputs)
    emb = output.last_hidden_state[0, 0, :].cpu().numpy()

    scores = cos(emb[np.newaxis, :], corpus_embeddings)[0]
    ranked_idx = np.argsort(-scores)
    if RETRIEVAL_TOP_K is None:
        top_k = ranked_idx
    else:
        top_k = ranked_idx[: int(RETRIEVAL_TOP_K)]
    curr_results = [(id2corpus_id[i], scores[i]) for i in top_k]
    scores = np.array([s for _, s in curr_results])
    curr_results = [(i, score_mapping(s, scores)) for i, s in curr_results]

    # construct candidate concepts
    topk_terms_dict = defaultdict(int)
    topk_topics_dict = defaultdict(int)
    N = len(curr_results) if CONCEPT_AGG_TOP_N is None else min(int(CONCEPT_AGG_TOP_N), len(curr_results))
    for rank, (corpusid, score) in enumerate(curr_results[:N]):
        for _, topic, _ in corpus_with_labels[str(corpusid)]['topics']:
            topk_topics_dict[topic] += 1
        for term in corpus_with_labels[str(corpusid)]['terms']:
            topk_terms_dict[term.lower()] += 1

    top_terms = [f'{t}, {c}' for t, c in sorted(topk_terms_dict.items(), key=lambda x: x[1], reverse=True)[:LLM_TOP_K_TERMS]]
    top_topics = [f'{t}, {c}' for t, c in sorted(topk_topics_dict.items(), key=lambda x: x[1], reverse=True)[:LLM_TOP_K_TOPICS]]

    top_papers = []
    for rank, (corpusid, _) in enumerate(curr_results[:min(LLM_TOP_PAPERS, len(curr_results))]):
        doc = corpus_with_labels[str(corpusid)]
        title = doc.get('title', '')
        abstract = doc.get('abstract', '')
        if title or abstract:
            paper_txt = f'Title: {title}\nAbstract: {abstract}'
        else:
            paper_txt = doc.get('text', '')
        top_papers.append(f'[{rank + 1}] {paper_txt}')

    llm_input = query_instruction % ('\n'.join(top_papers), '\n'.join(top_topics), '\n'.join(top_terms), query_text)
    llm_output = api_call(llm_input, model='gpt-4.1-mini')

    selected_concepts = []
    if '<ans>' in llm_output and '</ans>' in llm_output:
        selected_concepts = [t.strip().lower() for t in llm_output.split('<ans>')[1].split('</ans>')[0].split(', ')]
        selected_concepts = [t for t in selected_concepts if t in all_term2id]

    if len(selected_concepts) == 0:
        uniform_ranked = curr_results
        gaw_ranked = curr_results
    else:
        uniform_ranked = rerank_with_mode(curr_results, selected_concepts, mode='uniform')
        gaw_ranked = rerank_with_mode(curr_results, selected_concepts, mode='gaw')

    if DEBUG_SEMRANK:
        if len(selected_concepts) == 0:
            _n_empty_concepts += 1
        d10 = [str(x[0]) for x in curr_results[:10]]
        u10 = [str(x[0]) for x in uniform_ranked[:10]]
        g10 = [str(x[0]) for x in gaw_ranked[:10]]
        if d10 != u10:
            _n_top10_diff_u += 1
        if d10 != g10:
            _n_top10_diff_g += 1
        if query_id < DEBUG_LOG_FIRST_N:
            debug_rows.append({
                'query_id': query_id,
                'llm_output': (llm_output or '')[:DEBUG_LLM_SNIP],
                'selected_concepts': list(selected_concepts),
                'n_concepts': len(selected_concepts),
                'dense_top5': d10[:5],
                'uniform_top5': u10[:5],
            })
        if query_id == MANUAL_COMPARE_IDX:
            rel = list(ori_q['corpusids'])
            manual_snapshot = {
                'query_id': query_id,
                'n_relevant': len(rel),
                'qrels_head': rel[:25],
                'dense_top10': d10,
                'uniform_top10': u10,
                'gaw_top10': g10,
                'relevant_in_dense_top10': [i for i in d10 if i in set(rel)],
                'relevant_in_uniform_top10': [i for i in u10 if i in set(rel)],
            }

    if scoring_mode in ['uniform', 'both']:
        results_uniform.append({
            'query': query_text,
            'corpusids': ori_q['corpusids'],
            'retrieved': uniform_ranked,
        })

    if scoring_mode in ['gaw', 'both']:
        results_gaw.append({
            'query': query_text,
            'corpusids': ori_q['corpusids'],
            'retrieved': gaw_ranked,
        })

if scoring_mode == 'uniform':
    results = results_uniform
elif scoring_mode == 'gaw':
    results = results_gaw
else:
    results = {
        'uniform': results_uniform,
        'gaw': results_gaw,
    }

if DEBUG_SEMRANK:
    nq = len(raw_queries)
    print('\n=== SemRank LLM / rerank debug ===')
    print(f'queries={nq} | empty selected_concepts (pure dense fallback): {_n_empty_concepts} ({100*_n_empty_concepts/max(nq,1):.1f}%)')
    print(f'top-10 ranking differs from dense: uniform {_n_top10_diff_u}, gaw {_n_top10_diff_g}')
    for row in debug_rows:
        print(f"\n--- query_id={row['query_id']} | n_concepts={row['n_concepts']} ---")
        print('selected_concepts:', row['selected_concepts'][:40], '...' if len(row['selected_concepts']) > 40 else '')
        print('dense_top5:   ', row['dense_top5'])
        print('uniform_top5: ', row['uniform_top5'])
        print('llm_output (trunc):', row['llm_output'][:1200], '...' if len(row['llm_output']) > 1200 else '')
    if manual_snapshot is not None:
        m = manual_snapshot
        print(f"\n=== MANUAL_COMPARE_IDX={MANUAL_COMPARE_IDX} (query_id={m['query_id']}) ===")
        print(f"|qrels|={m['n_relevant']} (showing first 25 ids): {m['qrels_head']}")
        print('dense_top10:   ', m['dense_top10'])
        print('uniform_top10: ', m['uniform_top10'])
        print('gaw_top10:     ', m['gaw_top10'])
        print('relevant ∩ dense_top10:   ', m['relevant_in_dense_top10'])
        print('relevant ∩ uniform_top10: ', m['relevant_in_uniform_top10'])


100%|██████████| 34/34 [14:37<00:00, 25.81s/it]


=== SemRank LLM / rerank debug ===
queries=34 | empty selected_concepts (pure dense fallback): 34 (100.0%)
top-10 ranking differs from dense: uniform 0, gaw 0

--- query_id=0 | n_concepts=0 ---
selected_concepts: [] 
dense_top5:    ['10010426', '195699380', '940795', '1070708', '6493753']
uniform_top5:  ['10010426', '195699380', '940795', '1070708', '6493753']
llm_output (trunc):  

--- query_id=1 | n_concepts=0 ---
selected_concepts: [] 
dense_top5:    ['10014168', '934992', '1766004', '8819802', '16326127']
uniform_top5:  ['10014168', '934992', '1766004', '8819802', '16326127']
llm_output (trunc):  

--- query_id=2 | n_concepts=0 ---
selected_concepts: [] 
dense_top5:    ['10052042', '16447573', '5120787', '57570672', '10980859']
uniform_top5:  ['10052042', '16447573', '5120787', '57570672', '10980859']
llm_output (trunc):  

--- query_id=3 | n_concepts=0 ---
selected_concepts: [] 
dense_top5:    ['102353905', '2797612', '15359942', '6054133', '52960685']
uniform_top5:  ['102353905'

In [ ]:
if 'EVAL_KS' not in globals():
    EVAL_KS = (50, 100)
if isinstance(results, dict):
    e_uniform = evaluate_metrics(results['uniform'], EVAL_KS, verbose=False)
    e_gaw = evaluate_metrics(results['gaw'], EVAL_KS, verbose=False)
    e = {'uniform': e_uniform, 'gaw': e_gaw}
    print_recall_semrank_table(dataset, e_uniform, e_gaw)
else:
    e_other = evaluate_metrics(results, EVAL_KS, verbose=False)
    e = {'mode': e_other}
    print(f'\n=== {dataset} (queries={e_other["num_queries"]}) — {scoring_mode} ===')
    print(f'{"":14}  {"R@50":>10}  {"R@100":>10}')
    print(f'{scoring_mode:14}  {e_other["recall@k"][50]:>10.4f}  {e_other["recall@k"][100]:>10.4f}')


=== csfcube (queries=34) — SemRank ===
                      R@50       R@100
semrank+u           0.6816      0.7588
semrank_gaw         0.6816      0.7588

Paper Table 2 (CSFCube): SPECTER-v2 dense ~0.533 / ~0.686; +SemRank ~0.622 / ~0.760. Qrels: union of aspect files, relevance score ≥ 2 (matches Table 1 doc/query≈13.32).


# Task 2 Agentic Critique and Refine Second Pass

## Imports and config

In [ ]:
import importlib
import agent_utils
importlib.reload(agent_utils)
from agent_utils import (
    critique_instruction,
    run_critique_agent,
    parse_critique_output,
    build_top3_abstract_block,
    refine_selected_concepts,
)

AGENT_TOP_K_FOR_CRITIQUE = 3
AGENT_MAX_REFINED_CONCEPTS = 12
AGENT_RERANK_MODE = 'gaw'
AGENT_DEBUG_LOG_FIRST_N = 5
AGENT_LLM_SNIP = 3000

print('agent_utils loaded')
print('critique top-k abstracts:', AGENT_TOP_K_FOR_CRITIQUE)
print('rerank mode for refined pass:', AGENT_RERANK_MODE)

agent_utils loaded
critique top-k abstracts: 3
rerank mode for refined pass: gaw


## Agentic per-query loop

In [ ]:
# second pass: critique-and-refine on top of the existing weighted (gaw) first pass
results_baseline_gaw = []
results_agentic = []
agent_debug_rows = []

_n_empty_first_pass = 0
_n_empty_refined = 0
_n_top10_changed = 0

for query_id, ori_q in enumerate(tqdm(raw_queries)):
    query_text = ori_q['query']

    inputs = tokenizer(query_text, return_tensors="pt", truncation=True, max_length=512, return_token_type_ids=False).to(device)
    with torch.no_grad():
        output = encoder(**inputs)
    emb = output.last_hidden_state[0, 0, :].cpu().numpy()

    scores = cos(emb[np.newaxis, :], corpus_embeddings)[0]
    ranked_idx = np.argsort(-scores)
    if RETRIEVAL_TOP_K is None:
        top_k = ranked_idx
    else:
        top_k = ranked_idx[: int(RETRIEVAL_TOP_K)]
    curr_results = [(id2corpus_id[i], scores[i]) for i in top_k]
    score_arr = np.array([s for _, s in curr_results])
    curr_results = [(i, score_mapping(s, score_arr)) for i, s in curr_results]

    topk_terms_dict = defaultdict(int)
    topk_topics_dict = defaultdict(int)
    N = len(curr_results) if CONCEPT_AGG_TOP_N is None else min(int(CONCEPT_AGG_TOP_N), len(curr_results))
    for rank, (corpusid, score) in enumerate(curr_results[:N]):
        for _, topic, _ in corpus_with_labels[str(corpusid)]['topics']:
            topk_topics_dict[topic] += 1
        for term in corpus_with_labels[str(corpusid)]['terms']:
            topk_terms_dict[term.lower()] += 1

    top_terms_sorted  = sorted(topk_terms_dict.items(),  key=lambda x: x[1], reverse=True)[:LLM_TOP_K_TERMS]
    top_topics_sorted = sorted(topk_topics_dict.items(), key=lambda x: x[1], reverse=True)[:LLM_TOP_K_TOPICS]
    top_terms  = [f'{t}, {c}' for t, c in top_terms_sorted]
    top_topics = [f'{t}, {c}' for t, c in top_topics_sorted]

    # Candidate vocab passed to the critique so REFINED_CONCEPTS come from the indexed vocabulary
    candidate_vocab = [t for t, _ in top_topics_sorted] + [t for t, _ in top_terms_sorted]

    top_papers = []
    for rank, (corpusid, _) in enumerate(curr_results[:min(LLM_TOP_PAPERS, len(curr_results))]):
        doc = corpus_with_labels[str(corpusid)]
        title = doc.get('title', '')
        abstract = doc.get('abstract', '')
        if title or abstract:
            paper_txt = f'Title: {title}\nAbstract: {abstract}'
        else:
            paper_txt = doc.get('text', '')
        top_papers.append(f'[{rank + 1}] {paper_txt}')

    llm_input = query_instruction % ('\n'.join(top_papers), '\n'.join(top_topics), '\n'.join(top_terms), query_text)
    llm_output = api_call(llm_input, model='gpt-4.1-mini')

    selected_concepts = []
    if '<ans>' in llm_output and '</ans>' in llm_output:
        raw_parsed = [t.strip().lower() for t in llm_output.split('<ans>')[1].split('</ans>')[0].split(', ')]
        selected_concepts = [t for t in raw_parsed if t in all_term2id]
    else:
        raw_parsed = []

    if query_id < AGENT_DEBUG_LOG_FIRST_N:
        print(f'\n[query {query_id}] raw first-pass LLM output: {repr((llm_output or "(empty)")[:300])}')
        print(f'[query {query_id}] raw parsed concepts: {raw_parsed[:15]}')
        print(f'[query {query_id}] in-vocab selected:   {selected_concepts[:15]}')

    if len(selected_concepts) == 0:
        first_pass_ranked = curr_results
        _n_empty_first_pass += 1
    else:
        first_pass_ranked = rerank_with_mode(curr_results, selected_concepts, mode=AGENT_RERANK_MODE)

    results_baseline_gaw.append({
        'query': query_text,
        'corpusids': ori_q['corpusids'],
        'retrieved': first_pass_ranked,
    })

    top3_text = build_top3_abstract_block(first_pass_ranked, corpus_with_labels, k=AGENT_TOP_K_FOR_CRITIQUE)
    critique_output = run_critique_agent(api_call, query_text, selected_concepts, top3_text,
                                         candidate_vocab=candidate_vocab)
    critique_dict = parse_critique_output(critique_output)
    refined_concepts = refine_selected_concepts(
        selected_concepts, critique_dict, all_term2id, max_concepts=AGENT_MAX_REFINED_CONCEPTS
    )

    if len(refined_concepts) == 0:
        final_ranked = first_pass_ranked
        _n_empty_refined += 1
    else:
        final_ranked = rerank_with_mode(curr_results, refined_concepts, mode=AGENT_RERANK_MODE)

    results_agentic.append({
        'query': query_text,
        'corpusids': ori_q['corpusids'],
        'retrieved': final_ranked,
    })

    baseline_top10 = [str(x[0]) for x in first_pass_ranked[:10]]
    final_top10 = [str(x[0]) for x in final_ranked[:10]]
    if baseline_top10 != final_top10:
        _n_top10_changed += 1

    agent_debug_rows.append({
        'query_id': query_id,
        'query_text': query_text,
        'initial_selected_concepts': list(selected_concepts),
        'critique_output': (critique_output or '')[:AGENT_LLM_SNIP],
        'missing_concepts': critique_dict['missing'],
        'misaligned_concepts': critique_dict['misaligned'],
        'refined_critique_concepts': critique_dict['refined'],
        'final_refined_concepts': refined_concepts,
        'baseline_top10': baseline_top10,
        'final_top10': final_top10,
    })

    if query_id < AGENT_DEBUG_LOG_FIRST_N:
        print(f"\n--- query_id={query_id} ---")
        print('query:', query_text[:160])
        print('initial concepts:', selected_concepts[:20])
        print('raw critique output:', (critique_output or '(empty)')[:600])
        print('critique missing:   ', critique_dict['missing'][:20])
        print('critique misaligned:', critique_dict['misaligned'][:20])
        print('critique refined:   ', critique_dict['refined'][:20])
        print('final refined (in vocab):', refined_concepts[:20])
        print('top10 changed:', baseline_top10 != final_top10)
        print('baseline top10:', baseline_top10)
        print('agentic  top10:', final_top10)

print('\n=== Task 2 agentic loop summary ===')
print(f'queries={len(raw_queries)}')
print(f'empty first-pass selected_concepts: {_n_empty_first_pass}')
print(f'empty refined_concepts (fell back to first pass): {_n_empty_refined}')
print(f'top-10 changed by agentic refinement: {_n_top10_changed}')

  0%|          | 0/34 [00:00<?, ?it/s]


[query 0] raw first-pass LLM output: '(empty)'
[query 0] raw parsed concepts: []
[query 0] in-vocab selected:   []


  3%|▎         | 1/34 [00:51<28:18, 51.46s/it]


--- query_id=0 ---
query: And That's A Fact: Distinguishing Factual and Emotional Argumentation in Online Dialogue[SEP]We investigate the characteristics of factual and emotional argumen
initial concepts: []
raw critique output: (empty)
critique missing:    []
critique misaligned: []
critique refined:    []
final refined (in vocab): []
top10 changed: False
baseline top10: ['10010426', '195699380', '940795', '1070708', '6493753', '199776432', '6819967', '9312342', '22309708', '141720169']
agentic  top10: ['10010426', '195699380', '940795', '1070708', '6493753', '199776432', '6819967', '9312342', '22309708', '141720169']

[query 1] raw first-pass LLM output: '(empty)'
[query 1] raw parsed concepts: []
[query 1] in-vocab selected:   []


  6%|▌         | 2/34 [01:42<27:26, 51.45s/it]

...


--- query_id=3 ---
query: Document-Level $N$-ary Relation Extraction with Multiscale Representation Learning[SEP]Most information extraction methods focus on binary relations expressed w
initial concepts: []
raw critique output: (empty)
critique missing:    []
critique misaligned: []
critique refined:    []
final refined (in vocab): []
top10 changed: False
baseline top10: ['102353905', '2797612', '15359942', '6054133', '52960685', '189927788', '184487889', '3576631', '202541610', '12390812']
agentic  top10: ['102353905', '2797612', '15359942', '6054133', '52960685', '189927788', '184487889', '3576631', '202541610', '12390812']

[query 4] raw first-pass LLM output: '(empty)'
[query 4] raw parsed concepts: []
[query 4] in-vocab selected:   []


 15%|█▍        | 5/34 [04:17<24:51, 51.44s/it]


--- query_id=4 ---
query: Naturalizing a Programming Language via Interactive Learning[SEP]Our goal is to create a convenient natural language interface for performing well-specified but
initial concepts: []
raw critique output: (empty)
critique missing:    []
critique misaligned: []
critique refined:    []
final refined (in vocab): []
top10 changed: False
baseline top10: ['10695055', '12225249', '9955699', '9864100', '2705742', '6630126', '17953497', '126147940', '58822924', '9491703']
agentic  top10: ['10695055', '12225249', '9955699', '9864100', '2705742', '6630126', '17953497', '126147940', '58822924', '9491703']


100%|██████████| 34/34 [29:17<00:00, 51.70s/it]


=== Task 2 agentic loop summary ===
queries=34
empty first-pass selected_concepts: 34
empty refined_concepts (fell back to first pass): 34
top-10 changed by agentic refinement: 0


## Evaluation weighted baseline vs weighted plus agentic

In [ ]:
if 'EVAL_KS' not in globals():
    EVAL_KS = (50, 100)

e_baseline = evaluate_metrics(results_baseline_gaw, EVAL_KS, verbose=False)
e_agentic = evaluate_metrics(results_agentic, EVAL_KS, verbose=False)

rb = e_baseline['recall@k']
ra = e_agentic['recall@k']
hb = e_baseline['hit@k']
ha = e_agentic['hit@k']

print(f'\n=== {dataset} (queries={e_baseline["num_queries"]}) Task 2 agentic vs weighted baseline ===')
print(f'{"":18}  {"R@50":>10}  {"R@100":>10}  {"Hit@50":>10}  {"Hit@100":>10}')
print(f'{"weighted_gaw":18}  {rb[50]:>10.4f}  {rb[100]:>10.4f}  {hb[50]:>10.4f}  {hb[100]:>10.4f}')
print(f'{"weighted_gaw+agent":18}  {ra[50]:>10.4f}  {ra[100]:>10.4f}  {ha[50]:>10.4f}  {ha[100]:>10.4f}')
print(f'{"delta":18}  {ra[50]-rb[50]:>+10.4f}  {ra[100]-rb[100]:>+10.4f}  {ha[50]-hb[50]:>+10.4f}  {ha[100]-hb[100]:>+10.4f}')


=== csfcube (queries=34) Task 2 agentic vs weighted baseline ===
                          R@50       R@100      Hit@50     Hit@100
weighted_gaw            0.6816      0.7588      0.9706      0.9706
weighted_gaw+agent      0.6816      0.7588      0.9706      0.9706
delta                  +0.0000     +0.0000     +0.0000     +0.0000


## Save per-query agent debug log

In [ ]:
import csv

debug_csv_path = f'{dataset}_agent_debug.csv'
fieldnames = [
    'query_id',
    'query_text',
    'initial_selected_concepts',
    'missing_concepts',
    'misaligned_concepts',
    'refined_critique_concepts',
    'final_refined_concepts',
    'baseline_top10',
    'final_top10',
    'critique_output',
]

with open(debug_csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in agent_debug_rows:
        out = {}
        for k in fieldnames:
            v = row.get(k, '')
            if isinstance(v, list):
                out[k] = ' | '.join(str(x) for x in v)
            else:
                out[k] = v
        writer.writerow(out)

print(f'wrote {len(agent_debug_rows)} rows to {debug_csv_path}')

wrote 34 rows to csfcube_agent_debug.csv
